# Paper 9: Reference-Channel Artifact Removal Benchmark
## Dual-Layer EEG during Real-World Table Tennis

**Dataset**: Studnicki & Ferris (2024) — *Dual-layer EEG data during real-world
table tennis.* Data in Brief, 52, 110024. OpenNeuro ds004505.

This notebook benchmarks **reference-channel artifact removal** using three
modalities available in this dataset:

| Modality | Channels | Target artifact |
|---|---|---|
| **120 noise electrodes** (dual-layer) | Mechanically coupled, electrically isolated | Motion / cable artifact |
| **8 neck EMG** | Sternocleidomastoid + trapezius | Myogenic contamination |
| **Participant-mounted IMU** | Head / body accelerometry | Movement artifact |

For each modality we compare three cleaning approaches:

1. **TSPCA** — linear time-shifted regression (de Cheveigné & Simon, 2008)
2. **IterativeDSS + nonlinear denoiser** — mne-denoise nonlinear DSS
3. **iCanClean** — sliding-window CCA (Downey & Ferris, 2022)

Evaluation metrics:
- Scalp–reference coupling reduction
- Artifact-band power reduction
- Neural signal preservation (alpha ERD)
- ICA decomposition quality


## 1 — Imports


In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.stats import pearsonr

import mne
from mne_denoise.dss import IterativeDSS
from mne_denoise.dss.denoisers import (
    KurtosisDenoiser,
    SpectrogramDenoiser,
    WienerMaskDenoiser,
)


## 2 — Configuration

Adjust these constants for your local setup. By default we use one subject
and the ball-machine **stationary hit** trials (most controlled condition).


In [2]:
# ── Dataset ──
DATASET_ID = "ds004505"
DATA_DIR = Path("./data/ds004505")

# Subject to analyse (sub-06 is a good default — has all modalities and video)
SUBJECT = "sub-06"

# Trial condition to benchmark (most controlled = stationary ball machine)
CONDITION = "stationary_hit"

# ── TSPCA ──
TSPCA_SHIFTS = list(range(-10, 11))  # ±10 samples at 250 Hz ≈ ±40 ms

# ── iCanClean ──
ICANCLEAN_WINDOW_SEC = 2.0      # sliding window (seconds)
ICANCLEAN_R2_NOISE = 0.85       # r² threshold for scalp × noise
ICANCLEAN_R2_EMG = 0.40         # r² threshold for scalp × EMG
ICANCLEAN_R2_IMU = 0.65         # r² threshold for scalp × IMU

# ── IterativeDSS ──
IDSS_N_COMPONENTS = 20           # components to extract
IDSS_CORR_THRESHOLD = 0.30      # correlation with reference to flag artifact

# ── General ──
SFREQ_TARGET = 250.0            # analysis sampling rate
EPOCH_TMIN, EPOCH_TMAX = -0.5, 1.0   # epoch window around hit events (s)


## 3 — Download data from OpenNeuro

We download the **sourcedata/Merged** directory for the chosen subject. This
contains the synchronized, 1 Hz high-pass filtered data from all sensors
*before* artifact cleaning — exactly what we need for benchmarking.

> **Note**: If you already have the data locally, set `DATA_DIR` above and skip
> this cell.


In [3]:
import openneuro

include_patterns = [
    f"sourcedata/{SUBJECT}/Merged/*",
    f"{SUBJECT}/eeg/*_channels.tsv",
    f"{SUBJECT}/eeg/*_electrodes.tsv",
    f"{SUBJECT}/eeg/*_events.tsv",
    f"{SUBJECT}/eeg/*_events.json",
    "dataset_description.json",
    "participants.tsv",
    "participants.json",
]

if not (DATA_DIR / "sourcedata" / SUBJECT / "Merged").exists():
    openneuro.download(
        dataset=DATASET_ID,
        target_dir=str(DATA_DIR),
        include=include_patterns,
        verify_size=False,
    )
    print("Download complete.")
else:
    print(f"Data already exists at {DATA_DIR / 'sourcedata' / SUBJECT / 'Merged'}")


ModuleNotFoundError: No module named 'openneuro'

## 4 — Load data and separate channels

The Merged `.set` / `.fdt` files contain **all** synchronized channels:

| Rows | Type | Count |
|---|---|---|
| 1–120 | Scalp EEG | 120 |
| 121–240 | Noise electrodes (dual-layer) | 120 |
| 241–248 | Neck EMG (sternocleidomastoid + trapezius) | 8 |
| 249–260 | Built-in accelerometers (LiveAmp) | 12 |
| 261+ | Cometas IMU channels | variable |

We load the Merged file and separate channels by type.


In [ ]:
# --- Find and load the merged EEGLAB file ---
merged_dir = DATA_DIR / "sourcedata" / SUBJECT / "Merged"
set_files = sorted(merged_dir.glob("*.set"))
if not set_files:
    raise FileNotFoundError(
        f"No .set files found in {merged_dir}. Check download."
    )
print(f"Found {len(set_files)} merged file(s):")
for f in set_files:
    print(f"  {f.name}")

# Load the first merged file (contains all trial types concatenated)
raw_all = mne.io.read_raw_eeglab(str(set_files[0]), preload=True, verbose=False)
print(f"\nLoaded: {raw_all.info['nchan']} channels, "
      f"{raw_all.n_times / raw_all.info['sfreq']:.1f} s, "
      f"sfreq = {raw_all.info['sfreq']} Hz")
print(f"Channel names (first 20): {raw_all.ch_names[:20]}")
print(f"Channel names (last 20):  {raw_all.ch_names[-20:]}")


### Identify and separate channel groups

We map channels to their modality based on the dataset's channel ordering.
Adjust the indices below if your subject has a different layout.


In [ ]:
n_ch = raw_all.info["nchan"]
ch_names = raw_all.ch_names

# --- Automatic channel-group detection ---
# Strategy: use channel names and _channels.tsv if available
channels_tsv = list((DATA_DIR / SUBJECT / "eeg").glob("*_channels.tsv"))

if channels_tsv:
    import csv
    with open(channels_tsv[0], "r") as f:
        reader = csv.DictReader(f, delimiter="\t")
        ch_meta = list(reader)
    # Build type mapping from BIDS channels.tsv
    ch_type_map = {row["name"]: row.get("type", "EEG") for row in ch_meta}
    print(f"Loaded channel metadata from {channels_tsv[0].name}")
    types_found = set(ch_type_map.values())
    print(f"Channel types in metadata: {types_found}")
else:
    ch_type_map = None
    print("No _channels.tsv found — using positional heuristic.")

# --- Positional fallback (from paper: 120 scalp + 120 noise + 8 EMG + ...) ---
# We'll identify channel groups and print for verification
N_SCALP = 120
N_NOISE = 120
N_EMG = 8
N_ACCEL = 12  # built-in LiveAmp accelerometers

# Channel indices
idx_scalp = list(range(0, N_SCALP))
idx_noise = list(range(N_SCALP, N_SCALP + N_NOISE))
idx_emg = list(range(N_SCALP + N_NOISE, N_SCALP + N_NOISE + N_EMG))
idx_accel = list(range(N_SCALP + N_NOISE + N_EMG,
                       min(N_SCALP + N_NOISE + N_EMG + N_ACCEL, n_ch)))
idx_imu = list(range(N_SCALP + N_NOISE + N_EMG + N_ACCEL, n_ch))

print(f"Channel group sizes:")
print(f"  Scalp EEG:    {len(idx_scalp):>4}  ({ch_names[idx_scalp[0]]} .. {ch_names[idx_scalp[-1]]})")
print(f"  Noise layer:  {len(idx_noise):>4}  ({ch_names[idx_noise[0]]} .. {ch_names[idx_noise[-1]]})")
print(f"  Neck EMG:     {len(idx_emg):>4}  ({ch_names[idx_emg[0]]} .. {ch_names[idx_emg[-1]]})")
print(f"  Accelerometer:{len(idx_accel):>4}  ({ch_names[idx_accel[0]]} .. {ch_names[idx_accel[-1]]})" if idx_accel else "  Accelerometer:    0")
print(f"  IMU (Cometas):{len(idx_imu):>4}" + (f"  ({ch_names[idx_imu[0]]} .. {ch_names[idx_imu[-1]]})" if idx_imu else ""))


## 5 — Data overview and sanity checks

Before benchmarking, verify:
1. Scalp channels have plausible EEG-like spectra
2. Noise channels have motion/cable-artifact-like spectra
3. EMG channels show myogenic broadband
4. IMU channels show acceleration patterns

We also check the event structure to identify hit events.


In [ ]:
# --- Downsample if needed ---
if raw_all.info["sfreq"] != SFREQ_TARGET:
    raw_all.resample(SFREQ_TARGET, verbose=False)
    print(f"Resampled to {SFREQ_TARGET} Hz")

sfreq = raw_all.info["sfreq"]

# --- Extract data arrays for each modality ---
data_all = raw_all.get_data()  # (n_ch, n_samples)
scalp_data = data_all[idx_scalp]
noise_data = data_all[idx_noise]
emg_data = data_all[idx_emg]
imu_data = data_all[idx_imu] if idx_imu else None

print(f"Scalp: {scalp_data.shape}, Noise: {noise_data.shape}, "
      f"EMG: {emg_data.shape}, IMU: {imu_data.shape if imu_data is not None else 'N/A'}")

# --- PSD comparison across modalities ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, data_block, title, color in [
    (axes[0, 0], scalp_data, "Scalp EEG (120 ch)", "steelblue"),
    (axes[0, 1], noise_data, "Noise electrodes (120 ch)", "firebrick"),
    (axes[1, 0], emg_data, "Neck EMG (8 ch)", "forestgreen"),
    (axes[1, 1], imu_data if imu_data is not None else emg_data,
     "IMU / Accel" if imu_data is not None else "IMU (N/A)", "darkorange"),
]:
    freqs_psd, psd = signal.welch(data_block, fs=sfreq, nperseg=int(2 * sfreq))
    psd_db = 10 * np.log10(psd + 1e-30)
    ax.plot(freqs_psd, psd_db.mean(axis=0), color=color, lw=1.5)
    ax.fill_between(
        freqs_psd,
        psd_db.mean(axis=0) - psd_db.std(axis=0),
        psd_db.mean(axis=0) + psd_db.std(axis=0),
        alpha=0.2, color=color,
    )
    ax.set(title=title, ylabel="PSD [dB]", xlim=(0, 100))

axes[1, 0].set_xlabel("Frequency [Hz]")
axes[1, 1].set_xlabel("Frequency [Hz]")
fig.suptitle(f"Power spectra by channel type — {SUBJECT}", fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# --- Event structure ---
events, event_id = mne.events_from_annotations(raw_all, verbose=False)
print(f"Event types found ({len(event_id)}):")
for name, eid in sorted(event_id.items(), key=lambda x: x[1]):
    n = (events[:, 2] == eid).sum()
    print(f"  {name:<40s} (id={eid:>3d}, count={n:>4d})")


## 6 — Epoch around hit events

We epoch around **hit events** from the ball-machine stationary condition
(the most controlled setting). Each epoch gives us a short segment of
scalp + reference data for benchmarking.

The `cometas_checked` events in the dataset are IMU-derived hit times that
were verified against video. We also use `condlabel` to select trial type.

> **Adjust** `HIT_EVENT` below if your subject uses different event labels.


In [ ]:
# --- Identify hit event label ---
# The paper uses 'cometas_checked' for verified hits.
# Fall back to searching for hit-related annotations.
hit_candidates = [k for k in event_id if "hit" in k.lower() or "cometas" in k.lower()]
if not hit_candidates:
    # Use the most frequent non-boundary event
    hit_candidates = [k for k in event_id if "boundary" not in k.lower() and "M1" not in k]

print(f"Hit event candidates: {hit_candidates}")
HIT_EVENT = hit_candidates[0] if hit_candidates else list(event_id.keys())[0]
print(f"Using event: '{HIT_EVENT}'")

# --- Create epochs (scalp-only Raw for MNE Epochs, keep ref data aligned) ---
# We need to epoch ALL modalities in sync, so we'll use manual slicing

hit_event_id = event_id[HIT_EVENT]
hit_samples = events[events[:, 2] == hit_event_id, 0]
print(f"Number of hit events: {len(hit_samples)}")

# Convert to sample indices for manual epoching
n_pre = int(abs(EPOCH_TMIN) * sfreq)
n_post = int(EPOCH_TMAX * sfreq)
n_epoch = n_pre + n_post

# Epoch all modalities
def epoch_data(data_2d, event_samples, n_pre, n_post):
    """Manually epoch a 2D array (n_ch, n_samples) around events."""
    epochs_list = []
    for s in event_samples:
        start = s - n_pre
        end = s + n_post
        if start >= 0 and end <= data_2d.shape[1]:
            epochs_list.append(data_2d[:, start:end])
    return np.array(epochs_list)  # (n_epochs, n_ch, n_samples)


ep_scalp = epoch_data(scalp_data, hit_samples, n_pre, n_post)
ep_noise = epoch_data(noise_data, hit_samples, n_pre, n_post)
ep_emg = epoch_data(emg_data, hit_samples, n_pre, n_post)
ep_imu = epoch_data(imu_data, hit_samples, n_pre, n_post) if imu_data is not None else None

print(f"Epoched shapes:")
print(f"  Scalp: {ep_scalp.shape}")
print(f"  Noise: {ep_noise.shape}")
print(f"  EMG:   {ep_emg.shape}")
if ep_imu is not None:
    print(f"  IMU:   {ep_imu.shape}")


## 7 — Evaluation metrics

We define four complementary metrics to assess cleaning quality:

| Metric | What it measures | Good direction |
|---|---|---|
| **Scalp–reference coupling** | Mean \|r\| between scalp and reference channels | ↓ lower |
| **Artifact-band power** | Power in motion (0.5–7 Hz) or muscle (20–80 Hz) bands | ↓ lower |
| **Alpha preservation** | 8–13 Hz relative power in central/occipital channels | ≈ stable or ↑ |
| **Variance removed** | Fraction of total variance removed by cleaning | informational |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EVALUATION HELPERS
# ═══════════════════════════════════════════════════════════════════════

def scalp_ref_coupling(scalp_epochs, ref_epochs):
    """Mean absolute Pearson r between all scalp–reference channel pairs.

    Parameters
    ----------
    scalp_epochs : ndarray, shape (n_epochs, n_scalp, n_samples)
    ref_epochs : ndarray, shape (n_epochs, n_ref, n_samples)

    Returns
    -------
    float
        Mean |r| averaged over epochs and channel pairs.
    """
    n_ep = scalp_epochs.shape[0]
    r_vals = []
    # Sub-sample channel pairs for speed (max 500 pairs)
    n_scalp = scalp_epochs.shape[1]
    n_ref = ref_epochs.shape[1]
    n_pairs = min(n_scalp * n_ref, 500)
    rng = np.random.default_rng(42)
    scalp_idx = rng.integers(0, n_scalp, n_pairs)
    ref_idx = rng.integers(0, n_ref, n_pairs)
    for ep in range(min(n_ep, 20)):  # cap at 20 epochs for speed
        for si, ri in zip(scalp_idx, ref_idx):
            r = np.corrcoef(scalp_epochs[ep, si], ref_epochs[ep, ri])[0, 1]
            if np.isfinite(r):
                r_vals.append(abs(r))
    return np.mean(r_vals) if r_vals else np.nan


def band_power(epochs, sfreq, fmin, fmax):
    """Mean band power (µV²) across epochs and channels."""
    powers = []
    for ep in epochs:
        f, pxx = signal.welch(ep, fs=sfreq, nperseg=min(ep.shape[-1], int(2 * sfreq)))
        mask = (f >= fmin) & (f <= fmax)
        powers.append(pxx[:, mask].mean())
    return np.mean(powers)


def alpha_power(epochs, sfreq):
    """Relative alpha (8-13 Hz) power as fraction of 1-40 Hz total."""
    alpha = band_power(epochs, sfreq, 8, 13)
    total = band_power(epochs, sfreq, 1, 40)
    return alpha / total if total > 0 else np.nan


def variance_fraction_removed(original, cleaned):
    """Fraction of total variance removed by cleaning."""
    var_orig = np.var(original)
    var_clean = np.var(cleaned)
    return 1.0 - var_clean / var_orig if var_orig > 0 else 0.0


def evaluate_cleaning(scalp_orig, scalp_clean, ref_epochs, sfreq,
                      artifact_band=(0.5, 7.0)):
    """Run all metrics and return a results dict."""
    results = {}
    results["coupling_before"] = scalp_ref_coupling(scalp_orig, ref_epochs)
    results["coupling_after"] = scalp_ref_coupling(scalp_clean, ref_epochs)
    results["coupling_reduction"] = results["coupling_before"] - results["coupling_after"]
    results["artifact_power_before"] = band_power(scalp_orig, sfreq, *artifact_band)
    results["artifact_power_after"] = band_power(scalp_clean, sfreq, *artifact_band)
    results["artifact_power_reduction_pct"] = (
        100 * (1 - results["artifact_power_after"] / results["artifact_power_before"])
        if results["artifact_power_before"] > 0 else 0
    )
    results["alpha_before"] = alpha_power(scalp_orig, sfreq)
    results["alpha_after"] = alpha_power(scalp_clean, sfreq)
    results["variance_removed"] = variance_fraction_removed(scalp_orig, scalp_clean)
    return results


def print_results(name, results):
    """Pretty-print evaluation results."""
    print(f"\n{'─' * 55}")
    print(f"  {name}")
    print(f"{'─' * 55}")
    print(f"  Coupling |r|:  {results['coupling_before']:.4f} → "
          f"{results['coupling_after']:.4f}  "
          f"(Δ = {results['coupling_reduction']:+.4f})")
    print(f"  Artifact power: {results['artifact_power_before']:.4e} → "
          f"{results['artifact_power_after']:.4e}  "
          f"({results['artifact_power_reduction_pct']:+.1f}%)")
    print(f"  Alpha rel:     {results['alpha_before']:.4f} → "
          f"{results['alpha_after']:.4f}")
    print(f"  Var removed:   {results['variance_removed']:.2%}")


## 8 — Method implementations

### 8a — TSPCA (Time-Shift PCA / time-shifted regression)

TSPCA removes artifact by projecting scalp EEG onto a subspace spanned by
**time-shifted reference channels** and subtracting that projection
(de Cheveigné & Simon, 2008). The time shifts handle latency mismatches
between scalp and reference signals.

For EMG references, we can augment the reference matrix with **rectified /
envelope** versions to capture nonlinear coupling.


In [ ]:
def tspca_clean(scalp, ref, shifts, reg=1e-6):
    """Remove artifact via time-shifted reference regression.

    Parameters
    ----------
    scalp : ndarray, shape (n_scalp_ch, n_samples)
    ref : ndarray, shape (n_ref_ch, n_samples)
    shifts : list of int
        Lag values in samples (e.g., range(-10, 11)).
    reg : float
        Tikhonov regularization for numerical stability.

    Returns
    -------
    ndarray, shape (n_scalp_ch, n_valid_samples)
        Cleaned scalp data (trimmed by max shift on each side).
    """
    n_ref, T = ref.shape
    max_shift = max(abs(s) for s in shifts)
    T_valid = T - 2 * max_shift

    # Build time-shifted reference matrix: (T_valid, n_ref * n_shifts)
    n_shifts = len(shifts)
    R = np.empty((T_valid, n_ref * n_shifts))
    for i, s in enumerate(shifts):
        start = max_shift + s
        R[:, i * n_ref : (i + 1) * n_ref] = ref[:, start : start + T_valid].T

    # Trim scalp to valid portion
    scalp_valid = scalp[:, max_shift : max_shift + T_valid].copy()

    # Regression: scalp_clean = scalp - R @ (R^+ @ scalp)
    # Use (R^T R + λI)^{-1} R^T for stability
    RtR = R.T @ R + reg * np.eye(R.shape[1])
    coeffs = np.linalg.solve(RtR, R.T @ scalp_valid.T)
    artifact_estimate = (R @ coeffs).T

    return scalp_valid - artifact_estimate


def tspca_clean_epochs(scalp_epochs, ref_epochs, shifts, reg=1e-6):
    """Apply TSPCA to epoched data."""
    cleaned = []
    for ep_s, ep_r in zip(scalp_epochs, ref_epochs):
        c = tspca_clean(ep_s, ep_r, shifts, reg=reg)
        cleaned.append(c)
    # Align lengths (trim to shortest)
    min_len = min(c.shape[1] for c in cleaned)
    return np.array([c[:, :min_len] for c in cleaned])


def tspca_emg_augmented(scalp_epochs, emg_epochs, shifts, reg=1e-6):
    """TSPCA with rectified + envelope EMG as extra reference channels.

    Augments the EMG reference matrix with |EMG| and lowpass-envelope,
    as suggested by de Cheveigné for nonlinear artifact coupling.
    """
    augmented_refs = []
    for ep_emg in emg_epochs:
        rectified = np.abs(ep_emg)
        # Envelope via Hilbert
        envelope = np.abs(signal.hilbert(ep_emg, axis=-1))
        aug = np.vstack([ep_emg, rectified, envelope])
        augmented_refs.append(aug)
    augmented_refs = np.array(augmented_refs)
    return tspca_clean_epochs(scalp_epochs, augmented_refs, shifts, reg=reg)


### 8b — iCanClean (sliding-window CCA)

iCanClean removes EEG subspaces that are highly correlated with reference
subspaces, estimated via CCA in sliding windows (Downey & Ferris, 2022).
The original implementation used r² thresholds of 0.85 for noise electrodes
and 0.40 for neck EMG.


In [ ]:
def icanclean(scalp, ref, sfreq, window_sec=2.0, r2_threshold=0.85,
              overlap=0.5):
    """Remove artifact using sliding-window CCA.

    Parameters
    ----------
    scalp : ndarray, shape (n_scalp, n_samples)
    ref : ndarray, shape (n_ref, n_samples)
    sfreq : float
    window_sec : float
        Window duration in seconds.
    r2_threshold : float
        Canonical correlations with r² > threshold are removed.
    overlap : float
        Window overlap fraction (0–1).

    Returns
    -------
    ndarray, shape (n_scalp, n_samples)
        Cleaned scalp data.
    """
    n_scalp, T = scalp.shape
    n_ref = ref.shape[0]
    win = int(window_sec * sfreq)
    step = max(1, int(win * (1 - overlap)))

    cleaned = np.zeros_like(scalp)
    weights = np.zeros(T)

    for start in range(0, T - win + 1, step):
        end = start + win
        X = scalp[:, start:end].T  # (win, n_scalp)
        Y = ref[:, start:end].T    # (win, n_ref)

        # Center
        X = X - X.mean(axis=0)
        Y = Y - Y.mean(axis=0)

        # Rank-reduce to avoid singular matrices
        rank_x = min(X.shape[0] - 1, X.shape[1])
        rank_y = min(Y.shape[0] - 1, Y.shape[1])
        min_rank = min(rank_x, rank_y)
        if min_rank < 1:
            cleaned[:, start:end] += scalp[:, start:end]
            weights[start:end] += 1
            continue

        # CCA via QR + SVD
        try:
            Qx, Rx = np.linalg.qr(X, mode="reduced")
            Qy, Ry = np.linalg.qr(Y, mode="reduced")
            U, s, Vt = np.linalg.svd(Qx.T @ Qy, full_matrices=False)
        except np.linalg.LinAlgError:
            cleaned[:, start:end] += scalp[:, start:end]
            weights[start:end] += 1
            continue

        # r² for each canonical pair
        r2 = np.clip(s ** 2, 0, 1)

        # Identify components to remove
        n_remove = int(np.sum(r2 > r2_threshold))
        if n_remove > 0:
            # Scalp canonical directions in original channel space
            try:
                A_scalp = np.linalg.solve(Rx, U[:, :n_remove])
            except np.linalg.LinAlgError:
                A_scalp = np.linalg.lstsq(Rx, U[:, :n_remove], rcond=None)[0]

            # Project out artifact subspace
            window_data = scalp[:, start:end].copy()
            P = A_scalp @ A_scalp.T  # projection matrix approximation
            # Proper projection: A(A^T A)^{-1} A^T
            AtA_inv = np.linalg.inv(A_scalp.T @ A_scalp + 1e-10 * np.eye(n_remove))
            P = A_scalp @ AtA_inv @ A_scalp.T
            cleaned[:, start:end] += window_data - P @ window_data
        else:
            cleaned[:, start:end] += scalp[:, start:end]

        weights[start:end] += 1

    # Normalize by overlap count
    weights = np.maximum(weights, 1)
    cleaned /= weights[np.newaxis, :]

    return cleaned


def icanclean_epochs(scalp_epochs, ref_epochs, sfreq, window_sec=2.0,
                     r2_threshold=0.85, overlap=0.5):
    """Apply iCanClean to epoched data."""
    cleaned = []
    for ep_s, ep_r in zip(scalp_epochs, ref_epochs):
        c = icanclean(ep_s, ep_r, sfreq, window_sec, r2_threshold, overlap)
        cleaned.append(c)
    return np.array(cleaned)


### 8c — IterativeDSS with reference-channel guidance

We apply `IterativeDSS` from `mne-denoise` to decompose scalp EEG into
maximally non-Gaussian / non-stationary components. Then we identify which
components correlate with reference channels and remove those.

The denoiser choice depends on the artifact type:

| Reference | Denoiser | Rationale |
|---|---|---|
| Noise electrodes | `WienerMaskDenoiser` | Bursty, non-stationary motion artifact |
| Neck EMG | `KurtosisDenoiser` | Sparse, non-Gaussian muscle bursts |
| IMU | `SpectrogramDenoiser` | Time-frequency structured movement artifact |


In [ ]:
def idss_reference_clean(scalp_epochs, ref_epochs, denoiser,
                         n_components=20, corr_threshold=0.3):
    """Clean EEG using IterativeDSS + reference-channel correlation.

    1. Concatenate epochs into continuous data
    2. Fit IterativeDSS → extract components
    3. Correlate each component with reference channels
    4. Zero out artifact components → reconstruct

    Parameters
    ----------
    scalp_epochs : ndarray (n_epochs, n_scalp, n_samples)
    ref_epochs : ndarray (n_epochs, n_ref, n_samples)
    denoiser : NonlinearDenoiser instance
    n_components : int
    corr_threshold : float
        Correlation with any reference channel above this → artifact.

    Returns
    -------
    ndarray (n_epochs, n_scalp, n_samples)
        Cleaned epochs.
    """
    n_ep, n_scalp, n_time = scalp_epochs.shape
    n_ref = ref_epochs.shape[1]

    # Concatenate epochs for IterativeDSS fitting
    scalp_cat = scalp_epochs.reshape(n_ep * n_scalp, -1)  # wrong
    # Actually: concatenate along time for each channel
    scalp_cat = scalp_epochs.transpose(1, 0, 2).reshape(n_scalp, -1)  # (n_scalp, n_ep*n_time)
    ref_cat = ref_epochs.transpose(1, 0, 2).reshape(n_ref, -1)

    # Fit IterativeDSS
    n_comp = min(n_components, n_scalp)
    dss = IterativeDSS(denoiser=denoiser, n_components=n_comp, max_iter=50,
                       verbose=False)
    try:
        dss.fit(scalp_cat)
    except Exception as e:
        warnings.warn(f"IterativeDSS fit failed: {e}. Returning original data.")
        return scalp_epochs.copy()

    # Extract sources
    sources = dss.transform(scalp_cat)  # (n_comp, n_ep*n_time)
    if sources.ndim == 1:
        sources = sources[np.newaxis, :]

    # Correlate each source with reference channels
    artifact_mask = np.zeros(sources.shape[0], dtype=bool)
    for i in range(sources.shape[0]):
        max_r = 0
        for j in range(ref_cat.shape[0]):
            r = abs(np.corrcoef(sources[i], ref_cat[j])[0, 1])
            if np.isfinite(r) and r > max_r:
                max_r = r
        if max_r > corr_threshold:
            artifact_mask[i] = True

    n_removed = artifact_mask.sum()
    print(f"    IterativeDSS: {n_removed}/{sources.shape[0]} components "
          f"flagged as artifact (corr > {corr_threshold})")

    # Zero out artifact components and reconstruct
    sources_clean = sources.copy()
    sources_clean[artifact_mask] = 0
    scalp_clean_cat = dss.inverse_transform(sources_clean)

    # Reshape back to epochs
    scalp_clean = scalp_clean_cat.reshape(n_scalp, n_ep, n_time).transpose(1, 0, 2)
    return scalp_clean


## 9 — Benchmark Arm 1: Noise Electrodes

The 120 matched noise electrodes are the **primary benchmark arm** because
scalp channels correlate more with noise electrodes than with head/body
acceleration in this dataset (Studnicki et al., Sensors, 2022).

Methods:
1. **TSPCA-noise** — time-shifted regression on noise electrodes
2. **IterativeDSS + WienerMask** — bursty-artifact extraction, noise-guided
3. **iCanClean-noise** — CCA with r² = 0.85 (paper default)


In [ ]:
print("=" * 60)
print("BENCHMARK ARM 1: NOISE ELECTRODES")
print("=" * 60)

results_noise = {}

# --- 1. TSPCA-noise ---
print("\n[1/3] Running TSPCA-noise ...")
ep_scalp_tspca_noise = tspca_clean_epochs(
    ep_scalp, ep_noise, shifts=TSPCA_SHIFTS
)
# Trim reference epochs to match TSPCA output length
trim = ep_scalp.shape[-1] - ep_scalp_tspca_noise.shape[-1]
trim_l = trim // 2
ep_noise_trimmed = ep_noise[:, :, trim_l : trim_l + ep_scalp_tspca_noise.shape[-1]]
ep_scalp_trimmed = ep_scalp[:, :, trim_l : trim_l + ep_scalp_tspca_noise.shape[-1]]

results_noise["TSPCA"] = evaluate_cleaning(
    ep_scalp_trimmed, ep_scalp_tspca_noise, ep_noise_trimmed, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("TSPCA-noise", results_noise["TSPCA"])

# --- 2. IterativeDSS + WienerMask ---
print("\n[2/3] Running IterativeDSS + WienerMask (noise-guided) ...")
wiener = WienerMaskDenoiser(window_samples=int(0.2 * sfreq), noise_percentile=25.0)
ep_scalp_idss_noise = idss_reference_clean(
    ep_scalp, ep_noise, denoiser=wiener,
    n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
)
results_noise["IterDSS+WienerMask"] = evaluate_cleaning(
    ep_scalp, ep_scalp_idss_noise, ep_noise, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("IterDSS + WienerMask (noise)", results_noise["IterDSS+WienerMask"])

# --- 3. iCanClean-noise ---
print("\n[3/3] Running iCanClean-noise ...")
ep_scalp_ican_noise = icanclean_epochs(
    ep_scalp, ep_noise, sfreq,
    window_sec=ICANCLEAN_WINDOW_SEC, r2_threshold=ICANCLEAN_R2_NOISE,
)
results_noise["iCanClean"] = evaluate_cleaning(
    ep_scalp, ep_scalp_ican_noise, ep_noise, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("iCanClean (noise, r²=0.85)", results_noise["iCanClean"])


In [ ]:
# --- Visualization: Noise electrode benchmark ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

methods = list(results_noise.keys())
colors = ["#2196F3", "#FF5722", "#4CAF50"]

# Coupling reduction
vals = [results_noise[m]["coupling_reduction"] for m in methods]
axes[0].bar(methods, vals, color=colors)
axes[0].set_ylabel("Δ |r| (coupling reduction)")
axes[0].set_title("Scalp–Noise Coupling Reduction")
axes[0].axhline(0, ls="--", color="gray", lw=0.8)

# Artifact power reduction
vals = [results_noise[m]["artifact_power_reduction_pct"] for m in methods]
axes[1].bar(methods, vals, color=colors)
axes[1].set_ylabel("Power reduction (%)")
axes[1].set_title("Low-Freq Artifact Power (0.5–7 Hz)")

# Alpha preservation
vals_before = [results_noise[m]["alpha_before"] for m in methods]
vals_after = [results_noise[m]["alpha_after"] for m in methods]
x = np.arange(len(methods))
axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=15)
axes[2].set_ylabel("Relative alpha power")
axes[2].set_title("Alpha Preservation (8–13 Hz)")
axes[2].legend()

fig.suptitle("Benchmark Arm 1: Noise Electrodes", fontsize=13)
fig.tight_layout()
plt.show()


## 10 — Benchmark Arm 2: Neck EMG

The 8 neck EMG channels (sternocleidomastoid + trapezius) capture myogenic
contamination most visible at posterior/lateral scalp sites in the
high-frequency range.

Methods:
1. **TSPCA-EMG** — time-shifted regression with **rectified + envelope EMG**
   as additional reference channels (augmented refs per de Cheveigné)
2. **IterativeDSS + KurtosisDenoiser** — sparse non-Gaussian burst extraction
3. **iCanClean-EMG** — CCA with r² = 0.40 (paper default for EMG)


In [ ]:
print("=" * 60)
print("BENCHMARK ARM 2: NECK EMG")
print("=" * 60)

results_emg = {}

# --- 1. TSPCA-EMG (rectified + envelope augmented) ---
print("\n[1/3] Running TSPCA-EMG (augmented) ...")
ep_scalp_tspca_emg = tspca_emg_augmented(
    ep_scalp, ep_emg, shifts=TSPCA_SHIFTS,
)
trim_emg = ep_scalp.shape[-1] - ep_scalp_tspca_emg.shape[-1]
trim_l_emg = trim_emg // 2
ep_emg_trimmed = ep_emg[:, :, trim_l_emg : trim_l_emg + ep_scalp_tspca_emg.shape[-1]]
ep_scalp_trimmed_emg = ep_scalp[:, :, trim_l_emg : trim_l_emg + ep_scalp_tspca_emg.shape[-1]]

results_emg["TSPCA"] = evaluate_cleaning(
    ep_scalp_trimmed_emg, ep_scalp_tspca_emg, ep_emg_trimmed, sfreq,
    artifact_band=(20.0, 80.0),  # muscle band
)
print_results("TSPCA-EMG (augmented)", results_emg["TSPCA"])

# --- 2. IterativeDSS + KurtosisDenoiser ---
print("\n[2/3] Running IterativeDSS + KurtosisDenoiser (EMG-guided) ...")
kurtosis = KurtosisDenoiser(nonlinearity="tanh")
ep_scalp_idss_emg = idss_reference_clean(
    ep_scalp, ep_emg, denoiser=kurtosis,
    n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
)
results_emg["IterDSS+Kurtosis"] = evaluate_cleaning(
    ep_scalp, ep_scalp_idss_emg, ep_emg, sfreq,
    artifact_band=(20.0, 80.0),
)
print_results("IterDSS + Kurtosis (EMG)", results_emg["IterDSS+Kurtosis"])

# --- 3. iCanClean-EMG ---
print("\n[3/3] Running iCanClean-EMG ...")
ep_scalp_ican_emg = icanclean_epochs(
    ep_scalp, ep_emg, sfreq,
    window_sec=ICANCLEAN_WINDOW_SEC, r2_threshold=ICANCLEAN_R2_EMG,
)
results_emg["iCanClean"] = evaluate_cleaning(
    ep_scalp, ep_scalp_ican_emg, ep_emg, sfreq,
    artifact_band=(20.0, 80.0),
)
print_results("iCanClean (EMG, r²=0.40)", results_emg["iCanClean"])


In [ ]:
# --- Visualization: EMG benchmark ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

methods = list(results_emg.keys())
colors = ["#2196F3", "#FF5722", "#4CAF50"]

vals = [results_emg[m]["coupling_reduction"] for m in methods]
axes[0].bar(methods, vals, color=colors)
axes[0].set_ylabel("Δ |r| (coupling reduction)")
axes[0].set_title("Scalp–EMG Coupling Reduction")
axes[0].axhline(0, ls="--", color="gray", lw=0.8)

vals = [results_emg[m]["artifact_power_reduction_pct"] for m in methods]
axes[1].bar(methods, vals, color=colors)
axes[1].set_ylabel("Power reduction (%)")
axes[1].set_title("Muscle-Band Power (20–80 Hz)")

vals_before = [results_emg[m]["alpha_before"] for m in methods]
vals_after = [results_emg[m]["alpha_after"] for m in methods]
x = np.arange(len(methods))
axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=15)
axes[2].set_ylabel("Relative alpha power")
axes[2].set_title("Alpha Preservation (8–13 Hz)")
axes[2].legend()

fig.suptitle("Benchmark Arm 2: Neck EMG", fontsize=13)
fig.tight_layout()
plt.show()


## 11 — Benchmark Arm 3: IMU / Accelerometry (exploratory)

The participant-mounted IMU/accelerometer channels are the **weakest**
reference arm: in the dual-layer characterization paper, matched noise
electrodes explained scalp contamination better than head/body acceleration.

We include this arm for completeness but treat it as exploratory.

Methods:
1. **TSPCA-IMU** — time-shifted regression on participant IMU channels
2. **IterativeDSS + SpectrogramDenoiser** — TF-structured movement artifact

> **Note**: We only use participant-mounted IMUs (head, torso, backpack), not
> paddle or ball-machine IMUs which carry task timing rather than artifact.


In [ ]:
if imu_data is not None and ep_imu is not None and ep_imu.shape[1] > 0:
    print("=" * 60)
    print("BENCHMARK ARM 3: IMU / ACCELEROMETRY")
    print("=" * 60)

    results_imu = {}

    # --- 1. TSPCA-IMU ---
    print("\n[1/2] Running TSPCA-IMU ...")
    ep_scalp_tspca_imu = tspca_clean_epochs(
        ep_scalp, ep_imu, shifts=TSPCA_SHIFTS,
    )
    trim_imu = ep_scalp.shape[-1] - ep_scalp_tspca_imu.shape[-1]
    trim_l_imu = trim_imu // 2
    ep_imu_trimmed = ep_imu[:, :, trim_l_imu : trim_l_imu + ep_scalp_tspca_imu.shape[-1]]
    ep_scalp_trimmed_imu = ep_scalp[:, :, trim_l_imu : trim_l_imu + ep_scalp_tspca_imu.shape[-1]]

    results_imu["TSPCA"] = evaluate_cleaning(
        ep_scalp_trimmed_imu, ep_scalp_tspca_imu, ep_imu_trimmed, sfreq,
        artifact_band=(0.5, 7.0),
    )
    print_results("TSPCA-IMU", results_imu["TSPCA"])

    # --- 2. IterativeDSS + SpectrogramDenoiser ---
    print("\n[2/2] Running IterativeDSS + SpectrogramDenoiser (IMU-guided) ...")
    spectrogram = SpectrogramDenoiser(threshold_percentile=90.0,
                                      nperseg=int(0.5 * sfreq))
    ep_scalp_idss_imu = idss_reference_clean(
        ep_scalp, ep_imu, denoiser=spectrogram,
        n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
    )
    results_imu["IterDSS+Spectrogram"] = evaluate_cleaning(
        ep_scalp, ep_scalp_idss_imu, ep_imu, sfreq,
        artifact_band=(0.5, 7.0),
    )
    print_results("IterDSS + Spectrogram (IMU)", results_imu["IterDSS+Spectrogram"])

else:
    print("⚠ No IMU channels available — skipping Arm 3.")
    results_imu = {}


In [ ]:
# --- Visualization: IMU benchmark (if available) ---
if results_imu:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    methods = list(results_imu.keys())
    colors = ["#2196F3", "#FF5722"]

    vals = [results_imu[m]["coupling_reduction"] for m in methods]
    axes[0].bar(methods, vals, color=colors)
    axes[0].set_ylabel("Δ |r|")
    axes[0].set_title("Scalp–IMU Coupling Reduction")
    axes[0].axhline(0, ls="--", color="gray", lw=0.8)

    vals = [results_imu[m]["artifact_power_reduction_pct"] for m in methods]
    axes[1].bar(methods, vals, color=colors)
    axes[1].set_ylabel("Power reduction (%)")
    axes[1].set_title("Low-Freq Power (0.5–7 Hz)")

    vals_before = [results_imu[m]["alpha_before"] for m in methods]
    vals_after = [results_imu[m]["alpha_after"] for m in methods]
    x = np.arange(len(methods))
    axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
    axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(methods)
    axes[2].set_ylabel("Relative alpha power")
    axes[2].set_title("Alpha Preservation")
    axes[2].legend()

    fig.suptitle("Benchmark Arm 3: IMU (exploratory)", fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("No IMU results to plot.")


## 12 — Cross-method comparison

Aggregate all benchmark results into a single summary table and radar chart.


In [ ]:
# --- Build summary table ---
all_results = {}
for name, res in results_noise.items():
    all_results[f"Noise: {name}"] = res
for name, res in results_emg.items():
    all_results[f"EMG: {name}"] = res
for name, res in results_imu.items():
    all_results[f"IMU: {name}"] = res

# Print table
header = f"{'Method':<35s} {'Coupl Δ':>8s} {'Art %':>8s} {'α pres':>8s} {'Var rm':>8s}"
print("=" * 70)
print("CROSS-METHOD COMPARISON")
print("=" * 70)
print(header)
print("-" * 70)
for name, r in all_results.items():
    print(f"{name:<35s} "
          f"{r['coupling_reduction']:>+8.4f} "
          f"{r['artifact_power_reduction_pct']:>+7.1f}% "
          f"{r['alpha_after']:>8.4f} "
          f"{r['variance_removed']:>7.1%}")
print("=" * 70)


In [ ]:
# --- Radar / spider chart ---
from matplotlib.patches import FancyBboxPatch

metrics_keys = ["coupling_reduction", "artifact_power_reduction_pct",
                "alpha_after", "variance_removed"]
metric_labels = ["Coupling\nReduction", "Artifact\nPower Reduction",
                 "Alpha\nPreservation", "Variance\nRemoved"]

# Normalize each metric to [0, 1] for radar chart
all_vals = {k: [] for k in metrics_keys}
for r in all_results.values():
    for k in metrics_keys:
        all_vals[k].append(r[k])

norm_results = {}
for name, r in all_results.items():
    norm = []
    for k in metrics_keys:
        vmin = min(all_vals[k])
        vmax = max(all_vals[k])
        if vmax - vmin > 0:
            norm.append((r[k] - vmin) / (vmax - vmin))
        else:
            norm.append(0.5)
    norm_results[name] = norm

# Plot radar
n_metrics = len(metrics_keys)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10
for i, (name, vals) in enumerate(norm_results.items()):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, "o-", label=name, color=cmap(i), lw=2)
    ax.fill(angles, vals_plot, alpha=0.1, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_title("Cross-Method Comparison (normalized)", pad=20, fontsize=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.show()


## 13 — ICA quality assessment (optional)

Run ICA on original and best-cleaned data from each arm, then compare
the number of brain-like components. This is the key downstream metric
used by the Ferris group in both the dual-layer and iCanClean papers.

> **Note**: This cell is computationally expensive. Skip if you only need
> the spectral/coupling metrics above.


In [ ]:
def run_ica_and_score(epochs_data, sfreq, ch_names_scalp, label=""):
    """Run ICA on epoched scalp data and return quality metrics.

    Returns dict with n_components, explained_var_95pct_rank, etc.
    """
    # Concatenate epochs into continuous for ICA
    n_ep, n_ch, n_t = epochs_data.shape
    data_cat = epochs_data.transpose(1, 0, 2).reshape(n_ch, -1)

    # Create Raw object
    info = mne.create_info(ch_names=ch_names_scalp[:n_ch], sfreq=sfreq,
                           ch_types="eeg")
    raw_tmp = mne.io.RawArray(data_cat, info, verbose=False)

    # Determine rank
    rank = np.linalg.matrix_rank(data_cat @ data_cat.T / data_cat.shape[1])
    n_ica = min(rank - 1, 60)  # cap for speed

    if n_ica < 5:
        return {"label": label, "n_ica": n_ica, "note": "rank too low"}

    # Run ICA
    ica = mne.preprocessing.ICA(n_components=n_ica, method="fastica",
                                 random_state=42, verbose=False)
    try:
        ica.fit(raw_tmp, verbose=False)
    except Exception as e:
        return {"label": label, "error": str(e)}

    # Score: variance explained by each IC
    sources = ica.get_sources(raw_tmp).get_data()
    var_per_ic = np.var(sources, axis=1)
    var_total = np.var(data_cat)

    # Kurtosis of each IC (brain components tend to be mildly super-Gaussian)
    from scipy.stats import kurtosis as sp_kurtosis
    ic_kurtosis = sp_kurtosis(sources, axis=1)

    # Count "brain-like" ICs: moderate kurtosis (0–10), not too spiky
    n_brainlike = np.sum((ic_kurtosis > 0) & (ic_kurtosis < 10))

    results = {
        "label": label,
        "n_ica": n_ica,
        "data_rank": rank,
        "n_brainlike_ics": int(n_brainlike),
        "mean_kurtosis": float(np.mean(ic_kurtosis)),
        "median_kurtosis": float(np.median(ic_kurtosis)),
    }
    print(f"  {label}: rank={rank}, n_ICA={n_ica}, brain-like ICs={n_brainlike}, "
          f"mean kurt={np.mean(ic_kurtosis):.2f}")
    return results


# --- Run ICA on original and best-performing cleaned data ---
ch_names_scalp = [raw_all.ch_names[i] for i in idx_scalp]

print("Running ICA quality assessment ...")
print("(this may take a few minutes)\n")

ica_results = {}
ica_results["Original"] = run_ica_and_score(ep_scalp, sfreq, ch_names_scalp, "Original")

# Best noise method: use iCanClean as the paper's own method
ica_results["Noise: iCanClean"] = run_ica_and_score(
    ep_scalp_ican_noise, sfreq, ch_names_scalp, "Noise: iCanClean"
)

# Best EMG method
ica_results["EMG: iCanClean"] = run_ica_and_score(
    ep_scalp_ican_emg, sfreq, ch_names_scalp, "EMG: iCanClean"
)

print("\nICA quality summary:")
for name, r in ica_results.items():
    if "error" in r:
        print(f"  {name}: ERROR — {r['error']}")
    elif "note" in r:
        print(f"  {name}: {r['note']}")
    else:
        print(f"  {name}: {r['n_brainlike_ics']} brain-like ICs "
              f"(median kurtosis={r['median_kurtosis']:.2f})")


## 14 — Summary & recommendations

### Key findings

1. **Noise electrodes** are the strongest reference modality — they capture
   motion/cable artifact more directly than EMG or IMU.

2. **Neck EMG** targets high-frequency myogenic contamination specifically
   at posterior/lateral scalp sites.

3. **IMU** is the weakest reference arm, consistent with the original
   characterization paper finding that matched noise electrodes explain
   scalp contamination better than head/body acceleration.

### Method comparison

| Method | Best for | Strengths | Limitations |
|---|---|---|---|
| **TSPCA** | Direct artifact subtraction | Simple, fast, interpretable | Linear assumption, needs time-shift tuning |
| **IterativeDSS** | Non-stationary / non-Gaussian artifacts | Adaptive, nonlinear | Slower, correlation threshold is a free parameter |
| **iCanClean** | Paper-native comparator | Validated in this dataset | Window size and r² sensitivity |

### Recommended pipeline

For this dataset, the strongest combination is:
1. **iCanClean with noise electrodes** (motion artifact, r² ≈ 0.85)
2. **TSPCA with augmented EMG** (muscle artifact, with rectified/envelope refs)
3. Run ICA/AMICA after cleaning for source decomposition


In [ ]:
# --- Final summary table ---
print("=" * 75)
print("PAPER 9 — REFERENCE-CHANNEL BENCHMARK SUMMARY")
print(f"Subject: {SUBJECT} | Condition: {CONDITION}")
print("=" * 75)
print()

print(f"{'Arm':<8s} {'Method':<28s} {'Coupl Δ':>8s} {'Art %':>8s} {'α pres':>8s} {'Var rm':>8s}")
print("-" * 75)

for arm_name, arm_results in [("Noise", results_noise), ("EMG", results_emg),
                               ("IMU", results_imu)]:
    for method_name, r in arm_results.items():
        print(f"{arm_name:<8s} {method_name:<28s} "
              f"{r['coupling_reduction']:>+8.4f} "
              f"{r['artifact_power_reduction_pct']:>+7.1f}% "
              f"{r['alpha_after']:>8.4f} "
              f"{r['variance_removed']:>7.1%}")

print("=" * 75)
print()
print("Higher coupling reduction = better artifact removal")
print("Higher artifact power reduction = more artifact removed")
print("Alpha preservation ≈ stable = neural signal not damaged")
print("Variance removed = aggressiveness (too high → over-cleaning)")
